# DataSage Analysis Report: large_test.csv
**Rows:** 60000 | **Columns:** 5
---
This professional-grade notebook presents automated exploratory data analysis (EDA), cleaning results, and advanced machine learning modeling executed by DataSage.


## 📂 1. Load Raw Data & Setup


**Load raw data**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

car = pd.read_csv('large_test.csv')
print(f'Raw shape: {car.shape}')
car.head()


**Inspect data types**


In [ ]:
print(car.dtypes)
print()
car.describe(include='all').T


### 1b. Duplicate Removal


**Check for duplicates**


In [ ]:
print(f'Duplicate rows: {car.duplicated().sum()}')  # 0


### 1f. Outlier Detection (IQR)

The IQR (interquartile range) method flags values below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`. DataSage flags them but does **not** remove them by default — the decision is yours.


**IQR outlier detection: 'val1' → 431 flagged**


In [ ]:
# 'val1' IQR outlier detection
q1, q3 = car['val1'].quantile(0.25), car['val1'].quantile(0.75)
iqr = q3 - q1
lower_bound = -2.697
upper_bound = 2.695
is_outlier_val1 = (car['val1'] < lower_bound) | (car['val1'] > upper_bound)
print(f"'val1': {is_outlier_val1.sum()} outliers (below -2.70 or above 2.69)")
# Uncomment to remove outliers instead of flagging:
# car = car[~is_outlier_val1]
# car.reset_index(drop=True, inplace=True)


### 1g. Save Cleaned Data


**Save cleaned data**


In [ ]:
car.reset_index(drop=True, inplace=True)
print(f'Final shape: {car.shape}')
remaining_nulls = car.isnull().sum()
if remaining_nulls.sum() > 0:
    print('Remaining nulls:')
    print(remaining_nulls[remaining_nulls > 0])
car.to_csv('cleaned_large_test.csv', index=False)
print(f'✅ Cleaned data saved to "cleaned_large_test.csv"')
car.head()


In [ ]:
# Assign the cleaned dataset to 'df' for the remaining analysis
df = car.copy()


## 2. Exploratory Data Analysis
### Univariate Distributions


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    df[numeric_cols].hist(bins=30, figsize=(15, 10), layout=(-1, 3), color='skyblue', edgecolor='black')
    plt.suptitle('Histograms of Numeric Columns', y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()


> **Observation:** These distributions indicate the spread and central tendency of numeric features.


### Top Correlations


- **id** vs **val2**: -0.016


- **id** vs **val1**: -0.008


- **id** vs **val3**: -0.002


- **val2** vs **val3**: 0.002


- **val1** vs **val2**: -0.002


In [ ]:
cols = ['val2', 'val1', 'id', 'val3']
corr = df[cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True)
plt.title('Correlation Heatmap')
plt.show()


> **Observation:** Strong correlations (close to 1 or -1) indicate features that move together.


## 3. Anomaly Detection
Anomalies were detected using Isolation Forest, which isolates outliers in the multidimensional space.


**Total Anomalies Flagged:** 750


In [ ]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.05, random_state=42)
numeric_df = df.select_dtypes(include=[np.number]).dropna()
if not numeric_df.empty:
    df['is_anomaly'] = iso.fit_predict(numeric_df)
    print(f'Found {(df["is_anomaly"] == -1).sum()} anomalies')
    
    # Visualize anomalies if there are at least 2 numeric columns
    if len(numeric_df.columns) >= 2:
        col_x = numeric_df.columns[0]
        col_y = numeric_df.columns[1]
        plt.figure(figsize=(8, 6))
        sns.scatterplot(data=df, x=col_x, y=col_y, hue='is_anomaly', palette={1: 'blue', -1: 'red'}, alpha=0.6)
        plt.title(f'Anomaly Detection: {col_x} vs {col_y}')
        plt.show()


> **Observation:** Red points (-1) signify statistical outliers that deviate significantly from the rest of the dataset.


## 4. Cluster Analysis
K-Means clustering was applied to identify natural groupings in the data.


Clustering skipped or not applicable.
